In [0]:
from pyspark.sql import functions as F

silver_updates = (spark.table("workspace.amazon_fullfilment_silver.orders_silver")
    .filter("is_current = true")
    .groupBy("order_id", "customer_id")
    .agg(
        F.min(F.when(F.col("status") == 'Initial', F.col("updated_timestamp"))).alias("initial_ts"),
        F.min(F.when(F.col("status") == 'Picking', F.col("updated_timestamp"))).alias("picking_ts"),
        F.min(F.when(F.col("status") == 'Shipped', F.col("updated_timestamp"))).alias("shipped_ts"),
        F.last("status").alias("latest_status")
    )
    # 1. Populate date_key (YYYYMMDD) from the initial timestamp
    .withColumn("date_key", F.date_format("initial_ts", "yyyyMMdd").cast("int"))
    
    # 2. Determine is_completed based on Shipped status
    .withColumn("is_completed", F.when(F.col("latest_status") == 'Shipped', True).otherwise(False))
    
    # 3. Calculate total_fulfillment_time (Initial to Shipped) in minutes
    # We use unix_timestamp to get seconds, then divide by 60
    .withColumn("total_fulfillment_time", 
        F.when(F.col("shipped_ts").isNotNull(), 
            (F.unix_timestamp("shipped_ts") - F.unix_timestamp("initial_ts")) / 60
        ).otherwise(None).cast("int")
    )
)

# Register the updated logic as the view
silver_updates.createOrReplaceTempView("updates_view1")

# 3. Perform the CUMULATIVE MERGE
spark.sql("""
MERGE INTO workspace.amazon_fullfilment_gold.fact_order_lifecycle AS target
USING updates_view1 AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN
  UPDATE SET 
    target.date_picking = COALESCE(target.date_picking, source.picking_ts),
    target.date_shipped = COALESCE(target.date_shipped, source.shipped_ts),
    target.current_status = source.latest_status,
    target.is_completed = source.is_completed,
    target.total_fulfillment_time = source.total_fulfillment_time,
    -- Calculate internal milestone durations
    target.minutes_to_pick = CAST((unix_timestamp(source.picking_ts) - unix_timestamp(target.date_initial)) / 60 AS INT),
    target.minutes_to_ship = CAST((unix_timestamp(source.shipped_ts) - unix_timestamp(source.picking_ts)) / 60 AS INT),
    target._last_updated_timestamp = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (
    order_id, customer_id, date_key, 
    date_initial, date_picking, date_shipped, 
    total_fulfillment_time, current_status, is_completed, _last_updated_timestamp
  )
  VALUES (
    source.order_id, source.customer_id, source.date_key, 
    source.initial_ts, source.picking_ts, source.shipped_ts, 
    source.total_fulfillment_time, source.latest_status, source.is_completed, current_timestamp()
  )""")